# ERDES: audit the supplied dataset splits

**Goal:** check the candidate non-RD/RD task's supplied training, validation, and test
assignments against official metadata before inspecting videos or training a model.

This notebook runs independently: no variables or outputs from
[setup_and_eda.ipynb](setup_and_eda.ipynb) are required. Start at section 1 in a fresh
**CPU** kernel and run one cell at a time. It needs pandas, small metadata and split
downloads, and no GPU or videos. It preserves the supplied assignments.

Outputs: partition counts, clip-overlap checks, source checksums, and the unresolved
patient/eye grouping limitation. These are dataset-integrity findings, not model results.


## 1. Check the runtime and choose storage

Use pandas from the managed Colab runtime, or follow [the notebook guide](README.md)
for a local uv environment. If its import fails, inspect the selected kernel before continuing.

Local inputs go under ignored `data/erdes/`; this notebook's results go under ignored
`outputs/split_audit/`. On Colab, these folders are under `/content` on the remote machine.
Downloads happen in the selected runtime, independently of any local files.


In [1]:
import hashlib
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

cwd = Path.cwd()
project_root = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()),
    None,
)
if project_root is None:
    if not Path("/content").is_dir():
        raise RuntimeError("Open this notebook from the project folder, or select a Colab kernel.")
    project_root = Path("/content")

data_dir = project_root / "data" / "erdes"
output_dir = project_root / "outputs" / "split_audit"
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Input folder:", data_dir)
print("Results folder:", output_dir)


Python: 3.13.16
pandas: 2.2.3
Input folder: /content/data/erdes
Results folder: /content/outputs/split_audit


## 2. Load and verify the release metadata

Reuse the metadata CSV if present, or download it from
[Zenodo v1](https://zenodo.org/records/18644370). Verify its published checksum
before reading it. We need the video paths and dataset classes to check the split CSVs.
These labels are ground truth supplied by the dataset, not model predictions.


In [2]:
metadata_url = "https://zenodo.org/records/18644370/files/erdes_metadata.csv?download=1"
expected_md5 = "95828fb5bdffdbf8173b5bcfc99e8148"
metadata_path = data_dir / "erdes_metadata.csv"

if not metadata_path.exists():
    with urlopen(metadata_url, timeout=60) as response:
        content = response.read()
    assert hashlib.md5(content).hexdigest() == expected_md5, "Metadata download checksum mismatch"
    metadata_path.write_bytes(content)

metadata_bytes = metadata_path.read_bytes()
assert hashlib.md5(metadata_bytes).hexdigest() == expected_md5, "Cached metadata checksum mismatch"
df = pd.read_csv(metadata_path)
assert {"clip_id", "file_path", "diagnostic_class"}.issubset(df.columns), "Missing metadata columns"
assert len(df) == 5381, "Unexpected release clip count"
assert not df[["clip_id", "file_path", "diagnostic_class"]].isna().any().any(), "Missing metadata identifiers or classes"
assert df["clip_id"].is_unique and df["file_path"].is_unique, "Repeated metadata identifiers"
assert set(df["diagnostic_class"]) == {"rd", "non_rd"}, "Unexpected diagnostic classes"
print(f"PASS: release metadata checksum and identifiers; {len(df):,} clips loaded")


PASS: release metadata checksum and identifiers; 5,381 clips loaded


## 3. Download the official split archive

Use [Zenodo v1](https://zenodo.org/records/18644370), the same release as our metadata.
`splits.zip` is about 65 kB; the videos are a separate download and are not needed here.
Keep downloaded inputs under `data/erdes/`, which Git ignores.

**Colab:** this means `/content/data/erdes/` on the remote runtime, not your computer's
project folder. This cell downloads what Colab needs directly; no local folder upload is required.

We inspect `non_rd_vs_rd` as a candidate task: **0 = non-RD** (normal or PVD), **1 = RD**.
This encoding follows the authors' [task definition at the inspected revision](https://github.com/OSUPCVLab/ERDES/blob/4af67e0b9f4667edbae75a64a819884ee14d7a3b/erdes/data/create_data_splits.py).
Auditing this task does not commit us to a final research question or model.
We read the three CSVs from the archive and save byte-for-byte copies for inspection;
we do not execute the authors' split-generation code or change any assignments.


In [3]:
import io
import zipfile

split_archive_url = "https://zenodo.org/records/18644370/files/splits.zip?download=1"
expected_split_md5 = "c5630e3f056ce358dcd384aaed946e64"
split_archive_path = data_dir / "splits.zip"

if not split_archive_path.exists():
    with urlopen(split_archive_url, timeout=60) as response:
        content = response.read()
    assert hashlib.md5(content).hexdigest() == expected_split_md5, "Split archive checksum mismatch"
    split_archive_path.write_bytes(content)

split_archive_bytes = split_archive_path.read_bytes()
assert hashlib.md5(split_archive_bytes).hexdigest() == expected_split_md5, "Cached split archive checksum mismatch"

task_name = "non_rd_vs_rd"
label_to_class = {0: "non_rd", 1: "rd"}
split_dir = data_dir / "splits" / task_name
split_dir.mkdir(parents=True, exist_ok=True)
splits = {}
split_sources = {}

with zipfile.ZipFile(io.BytesIO(split_archive_bytes)) as archive:
    for split_name in ("train", "val", "test"):
        member = f"splits/{task_name}/{split_name}.csv"
        csv_bytes = archive.read(member)
        local_csv = split_dir / f"{split_name}.csv"
        if local_csv.exists():
            assert local_csv.read_bytes() == csv_bytes, f"Existing {local_csv.name} differs from the release; inspect it"
        else:
            local_csv.write_bytes(csv_bytes)
        splits[split_name] = pd.read_csv(io.BytesIO(csv_bytes))
        split_sources[split_name] = {
            "archive_member": member,
            "sha256": hashlib.sha256(csv_bytes).hexdigest(),
        }

print("PASS: official split archive checksum and unchanged CSV copies")
print("Candidate task:", task_name, "| label mapping:", label_to_class)
print("Saved split CSVs:", split_dir)


PASS: official split archive checksum and unchanged CSV copies
Candidate task: non_rd_vs_rd | label mapping: {0: 'non_rd', 1: 'rd'}
Saved split CSVs: /content/data/erdes/splits/non_rd_vs_rd


## 4. Match splits to metadata and check clip overlap

Match the entire relative video path, rather than assuming a filename prefix identifies a patient.
Check required columns, missing values, repeated paths, metadata matches, and label agreement.
Then check that the partitions share no clips and together cover this candidate task's metadata.

Inspecting test membership and label counts is a dataset integrity check. We do not use
test videos, predictions, or performance to choose preprocessing, models, or thresholds.


In [4]:
metadata_by_path = df.set_index("file_path", verify_integrity=True)
split_paths = {}
split_summary_rows = []

for split_name, split_df in splits.items():
    assert {"path", "label"}.issubset(split_df.columns), f"Missing columns in {split_name}"
    assert not split_df[["path", "label"]].isna().any().any(), f"Missing path/label in {split_name}"
    assert not split_df.empty, f"Empty split: {split_name}"
    assert split_df["path"].is_unique, f"Duplicate clip paths in {split_name}"
    assert set(split_df["label"]) == set(label_to_class), f"Unexpected labels in {split_name}"

    paths = set(split_df["path"])
    unmatched = paths - set(metadata_by_path.index)
    assert not unmatched, f"{split_name}: {len(unmatched)} paths absent from metadata"
    observed_classes = split_df["path"].map(metadata_by_path["diagnostic_class"])
    expected_classes = split_df["label"].map(label_to_class)
    assert observed_classes.eq(expected_classes).all(), f"Class/label disagreement in {split_name}"

    split_paths[split_name] = paths
    rd_count = int(split_df["label"].eq(1).sum())
    split_summary_rows.append({
        "split": split_name,
        "clips": len(split_df),
        "non_rd": int(split_df["label"].eq(0).sum()),
        "rd": rd_count,
        "rd_percent": round(100 * rd_count / len(split_df), 2),
    })

overlap_rows = []
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    shared = len(split_paths[left] & split_paths[right])
    overlap_rows.append({"pair": f"{left}/{right}", "shared_clip_paths": shared})

overlap_table = pd.DataFrame(overlap_rows)
display(overlap_table)
assert overlap_table["shared_clip_paths"].eq(0).all(), "Clips appear in multiple partitions"
all_split_paths = set().union(*split_paths.values())
assert all_split_paths == set(metadata_by_path.index), "Split coverage differs from the metadata"

split_summary = pd.DataFrame(split_summary_rows).set_index("split")
display(split_summary)
print("PASS: unique clip paths, complete metadata coverage, label agreement, and no cross-split clip overlap")
print("Patient/eye independence and duplicate video content are NOT established by these checks.")


,pair,shared_clip_paths
0,train/val,0
1,train/test,0
2,val/test,0


,clips,non_rd,rd,rd_percent
split,,,,
train,3873,3511,362,9.35
val,431,391,40,9.28
test,1077,977,100,9.29


PASS: unique clip paths, complete metadata coverage, label agreement, and no cross-split clip overlap
Patient/eye independence and duplicate video content are NOT established by these checks.


## 5. Record the grouping limitation and save the audit

The release CSV contains clip IDs and video paths, but no explicit patient or eye identifier.
The accompanying [metadata description](https://zenodo.org/records/18644370/files/erdes_metadata.json?download=1)
describes `clip_id` only as a unique clip identifier. The inspected authors' split generator
stratifies video rows by label; it does not explicitly group patients or eyes.

**Paper follow-up:** [v2, Data Record](https://arxiv.org/html/2508.04735v2#S3) explicitly
states that splits are clip-level because patient identifiers were removed. Clips from
the same patient may cross partitions. Thus patient separation is not enforced;
actual patient-overlap counts remain unavailable. Filenames use random IDs and counters,
not a documented patient map. No shared clip paths does not establish patient independence.
Video-content duplicates have not been checked. Preserve the supplied splits and
carry this documented limitation into future evaluation reports.

Save aggregate results and source checksums below. These outputs are separate from the
metadata exploration notebook's summary and plot.


In [5]:
grouping_fields = [column for column in df.columns if "patient" in column.lower() or "eye" in column.lower()]
print("Explicit patient/eye columns in the metadata:", grouping_fields or "none")
grouping_limit = (
    "Patient/eye independence is unverified: the inspected release has no explicit patient/eye IDs; "
    "clip IDs and filename prefixes are not a verified grouping map. Video-content duplicates were not checked."
)
print(grouping_limit)

split_summary_path = output_dir / "split_summary.csv"
split_summary.to_csv(split_summary_path)
split_audit = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_release": "https://zenodo.org/records/18644370",
    "task": task_name,
    "label_to_class": label_to_class,
    "metadata_sha256": hashlib.sha256(metadata_bytes).hexdigest(),
    "split_archive_url": split_archive_url,
    "split_archive_md5": expected_split_md5,
    "split_archive_sha256": hashlib.sha256(split_archive_bytes).hexdigest(),
    "split_sources": split_sources,
    "task_definition_source": "https://github.com/OSUPCVLab/ERDES/blob/4af67e0b9f4667edbae75a64a819884ee14d7a3b/erdes/data/create_data_splits.py",
    "runtime": {"python": sys.version, "pandas": pd.__version__},
    "split_counts": split_summary.reset_index().to_dict(orient="records"),
    "clip_overlap": overlap_rows,
    "checks_completed": [
        "release archive checksum", "unchanged CSV copies", "within-split path uniqueness",
        "metadata path matching", "numeric label agreement", "cross-split clip path overlap", "full metadata coverage",
    ],
    "supplied_assignments_preserved": True,
    "patient_eye_independence": "unverified",
    "limitations": [grouping_limit, "Video files have not been inspected by this notebook."],
    "pending": ["verify patient/eye grouping", "training video inspection", "baseline design"],
}
split_audit_path = output_dir / "split_audit.json"
split_audit_path.write_text(json.dumps(split_audit, indent=2) + "\n", encoding="utf-8")
print("Saved counts:", split_summary_path)
print("Saved audit:", split_audit_path)


Explicit patient/eye columns in the metadata: none
Patient/eye independence is unverified: the inspected release has no explicit patient/eye IDs; clip IDs and filename prefixes are not a verified grouping map. Video-content duplicates were not checked.
Saved counts: /content/outputs/split_audit/split_summary.csv
Saved audit: /content/outputs/split_audit/split_audit.json


## Stop here and review the split audit

Explain the train/validation/test counts, 0/1 label mapping, and what clip-overlap
checks establish. The paper now confirms clip-level splitting and possible same-patient
overlap; actual patient-overlap counts cannot be measured from the released identifiers.
Preserve `split_summary.csv` and `split_audit.json` outside temporary Colab storage.

**Next:** open [video_inspection.ipynb](video_inspection.ipynb) to inspect a small,
reproducibly selected training sample. It runs independently and records video provenance.
Then agree on a baseline task, preprocessing, metrics, and compute budget.
Reserve final test evaluation until model development is complete.

Sources: [dataset release](https://zenodo.org/records/18644370) ·
[paper v2, Data Record](https://arxiv.org/html/2508.04735v2#S3).
